In [ ]:
import python_calamine
import pandas as pd
import numpy as np
from pathlib import Path
import xlsxwriter

In [ ]:
#must have raw .xlsx named "imported_data.xslx"

base_path = Path.cwd()
raw_data = base_path / "data" / "imported_data.xlsx"

In [ ]:
raw_data_df = pd.read_excel(raw_data, sheet_name="CAPWindModeling")
# raw_data_df.head(10)
raw_data_df = raw_data_df.drop(columns=["Unnamed: 38"])
#raw_data_df.iloc[:, -15:]



In [ ]:
out_df = pd.DataFrame()

out_df["ACCNTNUM"]    = raw_data_df["QUOTEID"]      # renamed
out_df["ACCNTNAME"]   = raw_data_df["Named Insured"].str.replace(",", "", regex=False)  # minus commas
out_df["LOCNUM"]      = raw_data_df["BLDGNO"]       # renamed
out_df["STREETNAME"]  = raw_data_df["STNAME"]       # renamed
out_df["CITY"]        = raw_data_df["CITY"]         # renamed
out_df["STATECODE"]   = raw_data_df["STATE"]           # renamed
out_df["POSTALCODE"]  = raw_data_df["ZIP5"]         # renamed
out_df["COUNTY"]      = raw_data_df["COUNTY"]       # repasted
out_df["CNTRYSCHEME"] = "ISO2A"                     # constant
out_df["CNTRYCODE"]   = "US"                        # constant
out_df["BLDGSCHEME"]  = "FIRE"                      # constant
out_df["OCCSCHEME"]    = "ATC"                              # constant
out_df["OCCTYPE"]    = raw_data_df["OCCPCL"]                              # constant
out_df["NUMSTORIES"] = raw_data_df["NOSTORIES"]            # renamed (replaces earlier "NO" mapping)
out_df["YEARBUILT"]  = "1/1/" + raw_data_df["YEARBUILT"].astype(str)   # prepend "1/1/"
out_df["CV1VAL"]     = raw_data_df["LOCBLDREPL"]          # renamed
out_df["LOCBLDDEDU"] = raw_data_df["LOCBLDDEDU"]          # unchanged
out_df["CV2VAL"]     = raw_data_df["LOCCNTREPL"]          # renamed
out_df["FLOORAREA"]  = raw_data_df["SQFEET"]              # renamed
out_df["BLDGCLASS"] = raw_data_df["CONSTCL"]            # renamed
out_df["INCEPTDATE"] = raw_data_df["Effective Date"]    # renamed
out_df["EXPIREDATE"] = raw_data_df["Expiration Date"]   # renamed
out_df["CONSTQUALI"] = raw_data_df["CONSTQUA"]        # unchanged
out_df["ROOFSYS"]    = raw_data_df["ROOFSYS"]           # unchanged
out_df["ROOFAGE"]    = raw_data_df["ROOFAGE"]           # unchanged
out_df["ROOFGEOM"]   = raw_data_df["ROOFGEO"]           # renamed
out_df["ROOFANCH"]   = raw_data_df["ROOFANC"]           # renamed
out_df["CLADSYS"]    = raw_data_df["CLADSYS"]           # unchanged
out_df["FOUNDSYS"]   = raw_data_df["FOUNDSYS"]          # unchanged
out_df["ROOFEQUIP"]  = raw_data_df["ROOFEQUI"]         # unchanged
out_df["CLADRATE"]   = raw_data_df["CLADRATE"]          # unchanged
out_df["RESISTOPEN"] = raw_data_df["RESISTOPEN"]        # unchanged

#out_df




In [ ]:
account_path = base_path / "output" / "Account.csv"
account_path.parent.mkdir(parents=True, exist_ok=True)


# 1. one representative row per unique account (+ deductibles)
accounts = (
    out_df.assign(
        _bld_ded=raw_data_df["LOCBLDDEDU"].values,
        _min_ded=raw_data_df["LOCBLDDEDU"].values,
    )
    .groupby("ACCNTNUM", as_index=False)
    .first()
)

# 2. expand each account into 4 rows
template = [
    ("_2pct", "SCS", 3),
    ("_2pct", "WS",  2),
    ("_3pct", "WS",  2),
    ("_5pct",  "WS",  2),
]
rows = []
for _, acc in accounts.iterrows():
    for suffix, policynum, policytype in template:
        rows.append({
            "ACCTNUM":    f"{acc['ACCNTNUM']}{suffix}",
            "ACCNTNAME":  acc["ACCNTNAME"],
            "INCEPTDATE": acc["INCEPTDATE"],
            "EXPIREDATE": acc["EXPIREDATE"],
            "POLICYNUM":  policynum,
            "POLICYTYPE": policytype,
            "BLANDEDAMT": acc["_bld_ded"] if policytype == 3 else np.nan,
            "MINDEDAMT":  acc["_min_ded"] if policytype == 2 else np.nan,
        })
account_df = pd.DataFrame(rows, columns=[
    "ACCTNUM", "ACCNTNAME", "INCEPTDATE", "EXPIREDATE",
    "POLICYNUM", "POLICYTYPE", "BLANDEDAMT", "MINDEDAMT",
])

# 3. sort + write
account_df_sorted = (
    account_df
    .assign(
        _policy_order=account_df["POLICYNUM"].map({"SCS": 0, "WS": 1}),
        _pct=account_df["ACCTNUM"].str.extract(r"_(\d+)pc", expand=False).astype(int),
        _accnt=pd.to_numeric(account_df["ACCTNUM"].str.rsplit("_", n=1).str[0], errors="coerce"),
    )
    .sort_values(["_policy_order", "_pct", "_accnt"], ascending=[True, True, True])
    .drop(columns=["_policy_order", "_pct", "_accnt"])
    .reset_index(drop=True)
)

account_df_sorted.to_csv(account_path, index=False)
print(f"Wrote {len(account_df_sorted)} rows to {account_path}")




In [ ]:
ded_amounts = pd.DataFrame({"DEDPCT": [2, 3, 5]})
suffix_map = {2: "2pct", 3: "3pct", 5: "5pct"}          # same suffixes as the account file

location_df = (
    out_df
    .merge(ded_amounts, how="cross")                    # every building x {2, 3, 5}
    .assign(
        ACCTNUM=lambda d: d["ACCNTNUM"].astype(str) + "_" + d["DEDPCT"].map(suffix_map),  # e.g. 5_2pct
        _accnt=lambda d: pd.to_numeric(d["ACCNTNUM"], errors="coerce"),
    )
    .sort_values(["DEDPCT", "_accnt", "LOCNUM"], ascending=True)
    .drop(columns="_accnt")
    .reset_index(drop=True)
)
# move ACCTNUM to be the first column


# rename the two CV columns
location_df = location_df.rename(columns={"CV1VAL": "WSCV1VAL", "CV2VAL": "WSCV2VAL"})

is_2pct = location_df["DEDPCT"] == 2

location_df["WSSITEDED"] = location_df["DEDPCT"] / 100            # ded pct as decimal: 2 -> 0.02
location_df["TOCV1VAL"] = location_df["WSCV1VAL"].where(is_2pct)  # WSCV1VAL only on 2pct rows, else blank
location_df["TOCV2VAL"] = location_df["WSCV2VAL"].where(is_2pct)  # WSCV2VAL only on 2pct rows, else blank
location_df = location_df[["ACCTNUM"] + [c for c in location_df.columns if c != "ACCTNUM"]]
location_df = location_df.drop(columns=["LOCBLDDEDU", "DEDPCT"])
location_df
# place the three new columns right after WSCV2VAL
new_cols = ["ACCNTNUM", "WSSITEDED", "TOCV1VAL", "TOCV2VAL"]
cols = [c for c in location_df.columns if c not in new_cols]
pos = cols.index("WSCV2VAL") + 1
location_df = location_df[cols[:pos] + new_cols + cols[pos:]]


# 1. fix the first column name: ACCTNUM -> ACCNTNUM
location_df.columns = ["ACCNTNUM"] + list(location_df.columns[1:])

# 2. drop the duplicate ACCNTNUM (keep only the first occurrence)
dup = (location_df.columns == "ACCNTNUM") & location_df.columns.duplicated()
location_df = location_df.loc[:, ~dup]

# 3. move BLDGCLASS to right after BLDGSCHEME
cols = [c for c in location_df.columns if c != "BLDGCLASS"]
pos = cols.index("BLDGSCHEME") + 1
cols.insert(pos, "BLDGCLASS")
location_df = location_df[cols]

location_path = base_path / "output" / "Location.csv"
location_path.parent.mkdir(parents=True, exist_ok=True)

location_df.to_csv(location_path, index=False)

print(f"Wrote {len(location_df)} rows to {location_path}")


